# Storm Hidden-Markov Model

In [17]:
import numpy as np

## Model

### Transition/transfer matrix
Latent environment state $L \in \{\mathrm{calm}, \mathrm{storm}\}$ with transition (transfer) matrix
$$
    T = 
    \begin{pmatrix}
    \Gamma_{\text{calm} \to \text{calm}} & \Gamma_{\text{calm} \to \text{storm}} \\
    \Gamma_{\text{storm} \to \text{calm}} & \Gamma_{\text{storm} \to \text{storm}}
    \end{pmatrix}.
$$
Let $a = \Gamma_{\text{calm} \to \text{storm}}$ and $b = \Gamma_{\text{storm} \to \text{calm}}$, then
$$
    T = 
    \begin{pmatrix}
    1-a & a\\
    b & 1-b
    \end{pmatrix},
$$
with $a \ll b \ll 1$. Eigenvalues of $T$ are
$$
\lambda_1 = 1, \quad \lambda_2 = 1 - a - b,
$$
with spectral gap $\Delta = \lambda_1 - \lambda_2$ and correlation length $\xi = \frac{1}{-\ln \lambda_2}$
$$
\Delta = a + b \\
\xi = \frac{1}{-\ln (1 - a - b)}.
$$
Stationary left (right) eigenstate $\bra{l}$ ($\ket{r}$) corresponding to $\lambda_1 = 1$ are
$$
    \bra{l} = 
    \frac{1}{a+b}\begin{pmatrix}
    b & a
    \end{pmatrix},
    \quad
    \ket{r} = 
    \begin{pmatrix}
    1 \\ 1
    \end{pmatrix},
$$
satisfying $\bra{l}T = \bra{l}, \ T\ket{r} = \ket{r}, \ \braket{l|r} = 1$.
$$

### Emission operator
Denote $p_\mathrm{calm}^{(x)}$ ($p_\mathrm{storm}^{(x)}$) as the probability of emitting Pauli operator $x$ when the environment transitions to the calm (storm) state. The emission operators are
$$
A^{(x)} = T\mathrm{diag}(p_\mathrm{calm}^{(x)}, p_\mathrm{storm}^{(x)}),
$$
where we have used the standard HMM to MPS tensor mapping. For stationary storm fraction $\pi_\mathrm{storm} = \frac{a}{a+b}$, the average marginal rate for Pauli $x$ is
$$
\bar{p}^{(x)} = (1 - \pi_\mathrm{storm})p_\mathrm{calm}^{(x)} + \pi_\mathrm{storm}p_\mathrm{storm}^{(x)}.
$$

### SE Kraus representation
Working backwards, the system-environment Kraus operators for our model can be written as
$$
K_{i\to j}^{(x)} = \sqrt{T_{ij}}\sqrt{p^{(x)}_j}\ket{j}\!\!\bra{i}_E \otimes \sigma_S^{(x)},
$$
which can be further dilated to a Stinespring isometry.

### Test HMM libraries

In [18]:
a = 0.01
b = 0.1

pi_a = a / (a + b)
pi_b = b / (a + b)

p_X_calm = 0
p_Y_calm = 0
p_Z_calm = 0
p_I_calm = 1

p_X_storm = 0.25
p_Y_storm = 0.25
p_Z_storm = 0.25
p_I_storm = 0.25

In [19]:
sample_length = 20
num_samples = 100

#### Promeganate
(No GPU accelerated sampling)

In [20]:
import torch
from pomegranate.distributions import Categorical
from pomegranate.hmm import DenseHMM

In [21]:
calm = Categorical([[p_I_calm, p_X_calm, p_Y_calm, p_Z_calm]])
storm = Categorical([[p_I_storm, p_X_storm, p_Y_storm, p_Z_storm]])

model = DenseHMM(
    distributions = [calm, storm],
    edges = [[1.0 - a, a], [b, 1.0 - b]],
    starts = [pi_b, pi_a],
    sample_length = sample_length,
)

In [22]:
%timeit model.sample(n=num_samples)[0].flatten()

103 ms ± 16 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Dynamax (Jax)

In [23]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax import vmap

from dynamax.hidden_markov_model import CategoricalHMM

In [24]:
# num_states: Number of hidden states
# emission_dim: Dimension of the emission space
# num_classes: Size of the discrete emission alphabet

hmm = CategoricalHMM(num_states=2, emission_dim=1, num_classes=4)
params, _ = hmm.initialize(
    initial_probs=jnp.array([pi_b, pi_a]),
    transition_matrix=jnp.array([[1.0 - a, a], [b, 1.0 - b]]),
    emission_probs=jnp.array([[p_I_calm, p_X_calm, p_Y_calm, p_Z_calm], [p_I_storm, p_X_storm, p_Y_storm, p_Z_storm]]).reshape(2, 1, 4)
    )

In [25]:
key = jr.PRNGKey(np.random.randint(0, 2**32))

Single sample

In [26]:
# Sample one sequence
hmm.sample(params, key=key, num_timesteps=sample_length)[1].flatten()

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

Batched sampling

In [27]:
keys = jr.split(jr.PRNGKey(1), num_samples)
%timeit vmap(lambda k: hmm.sample(params, k, sample_length))(keys)

508 ms ± 8.62 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
keys = jr.split(key, num_samples)
output = vmap(lambda k: hmm.sample(params, k, sample_length))(keys)

In [35]:
output[1].shape

(100, 20, 1)

Nice gains from GPU acceleration of HMM sampling for batched sampling